# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from webscrapper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [12]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
client = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://www.sanmina.com/")
links

['/',
 'https://www.sanmina.com/company-profile/',
 'https://www.sanmina.com/company-profile/management-team/',
 'https://www.sanmina.com/company-profile/environmental-policy/',
 'https://www.sanmina.com/company-profile/legal-information/',
 'https://www.sanmina.com/social-responsibility/',
 'https://www.sanmina.com/social-responsibility/health-safety/',
 'https://www.sanmina.com/social-responsibility/environment/',
 'https://www.sanmina.com/social-responsibility/ethics-governance/',
 'https://www.sanmina.com/social-responsibility/employees/',
 'https://www.sanmina.com/social-responsibility/community/',
 'https://www.sanmina.com/social-responsibility/awards/',
 'https://ir.sanmina.com/',
 'https://www.sanmina.com/media-center/',
 '/media-center/case-studies/',
 '/media-center/events/',
 '/media-center/media-coverage/',
 '/media-center/press-releases/',
 '/media-center/videos/',
 'https://www.sanmina.com/media-center/media-contacts/',
 'https://www.sanmina.com/partners/',
 'https://www.

 ## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://www.sanmina.com/"))


Here is the list of links on the website https://www.sanmina.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
https://www.sanmina.com/company-profile/
https://www.sanmina.com/company-profile/management-team/
https://www.sanmina.com/company-profile/environmental-policy/
https://www.sanmina.com/company-profile/legal-information/
https://www.sanmina.com/social-responsibility/
https://www.sanmina.com/social-responsibility/health-safety/
https://www.sanmina.com/social-responsibility/environment/
https://www.sanmina.com/social-responsibility/ethics-governance/
https://www.sanmina.com/social-responsibility/employees/
https://www.sanmina.com/social-responsibility/community/
https://www.sanmina.com/social-responsibility/awards/
https://ir.sanmina.com/
https://www.sanmina.com/media-center/
/media-center/case

In [13]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    return links    

In [14]:
select_relevant_links("https://www.sanmina.com/")

{'links': [{'type': 'about page',
   'url': 'https://www.sanmina.com/company-profile/'},
  {'type': 'management team page',
   'url': 'https://www.sanmina.com/company-profile/management-team/'},
  {'type': 'careers page', 'url': 'https://www.sanmina.com/careers/'},
  {'type': 'social responsibility page',
   'url': 'https://www.sanmina.com/social-responsibility/'},
  {'type': 'locations page', 'url': 'https://www.sanmina.com/locations/'},
  {'type': 'contact page', 'url': 'https://www.sanmina.com/contact/'},
  {'type': 'partners page', 'url': 'https://www.sanmina.com/partners/'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        # reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [17]:
select_relevant_links("https://www.sanmina.com/")

Selecting relevant links for https://www.sanmina.com/ by calling gpt-4o-mini
Found 8 relevant links


{'links': [{'type': 'about page',
   'url': 'https://www.sanmina.com/company-profile/'},
  {'type': 'management team page',
   'url': 'https://www.sanmina.com/company-profile/management-team/'},
  {'type': 'environmental policy page',
   'url': 'https://www.sanmina.com/company-profile/environmental-policy/'},
  {'type': 'social responsibility page',
   'url': 'https://www.sanmina.com/social-responsibility/'},
  {'type': 'careers page', 'url': 'https://www.sanmina.com/careers'},
  {'type': 'locations page', 'url': 'https://www.sanmina.com/locations/'},
  {'type': 'contact page', 'url': 'https://www.sanmina.com/contact/'},
  {'type': 'partners page', 'url': 'https://www.sanmina.com/partners/'}]}

In [18]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 9 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'community discussion page',
   'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
Updated
2 days ago
•
24.5k
•
7.26k
MiniMaxAI/MiniMax-H3-Image
Updated
3 days ago
•
518k
•
1.09k
Lightricks/LTX-2.5
Updated
Oct 11
•
2.78M
•
11.9k
Qwen/Qwen3.8-27B
Updated
7 days ago
•
6.76M
•


In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nSearch models, datasets, users...\nCtrl K\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nxtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic\nUpdated\n2 days ago\n

In [25]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [27]:
create_brochure("Sanmina", "https://www.sanmina.com/")

Selecting relevant links for https://www.sanmina.com/ by calling gpt-4o-mini
Found 8 relevant links


# Sanmina

**The manufacturer behind the brand.**

Sanmina designs, manufactures and repairs complex products for original equipment manufacturers (OEMs). Its end-to-end capabilities span product design and engineering, manufacturing, logistics and repair—helping customers bring products to market and support them throughout their lifecycle.

## What Sanmina does

- **Design and engineering:** Product development, engineering services and quick-turn prototyping.
- **Electronics manufacturing:** Printed circuit boards, PCB assembly and surface-mount technology (SMT), test services and new-product introduction.
- **Systems and mechanical manufacturing:** Enclosures, precision machining, welded frames, plastics and complete systems.
- **Specialized technologies:** Backplanes, cable assemblies, interconnect systems, optical, RF and microelectronics, and SSD and DRAM solutions.
- **Supply chain and lifecycle support:** Supply-chain and vendor management, logistics, aftermarket services, repair and maintenance.

## Industries served

Sanmina serves customers across **medical systems, communications networks, defense and aerospace, industrial, clean technology, computing and storage, multimedia, automotive, and oil and gas**.

## Working with customers

Sanmina’s broad capabilities let OEMs work with one partner across multiple stages of product development and support. Its focus areas include quality, customer satisfaction, performance excellence, and continuous improvement through **Lean and Six Sigma**. The company also highlights business continuity and quality certifications.

## Culture and responsibility

Sanmina’s website features programs and policies covering **employees, community, social responsibility, health and safety, the environment, and ethics and governance**. These reflect a focus on responsible operations and the people and communities connected to its work.

## Careers

Sanmina has a **Careers** section for people interested in joining the company. Its range of engineering, technology, manufacturing and support capabilities offers work connected to complex products and global customer programs. See Sanmina’s careers information for current opportunities.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        # reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("Sanmina", "https://www.sanmina.com/")

Selecting relevant links for https://www.sanmina.com/ by calling gpt-4o-mini
Found 5 relevant links


# Sanmina Corporation Brochure

## About Us
Sanmina Corporation is a premier provider of end-to-end design, manufacturing, logistics, and repair solutions for Original Equipment Manufacturers (OEMs). With our deep expertise across various industries, including medical systems, defense and aerospace, communications networks, and clean technology, we are dedicated to transforming complex concepts into successful realities.

## Our Services
At Sanmina, we pride ourselves on offering diverse services tailored to meet our customers' unique needs. Our capabilities include:
- **Design & Engineering**: From product design to engineering services, we focus on innovation and quality.
- **Manufacturing**: Our advanced manufacturing includes printed circuit boards (PCBs), enclosures, complex cable harnesses, and precision machining.
- **Supply Chain Management**: We ensure smooth operations with efficient vendor management and logistics.
- **Aftermarket Services**: Providing robust repair and maintenance support for longevity and customer satisfaction.

Our commitment to performance excellence is driven by Lean and Six Sigma principles, guaranteeing high-quality outputs and customer satisfaction.

## Company Culture
Sanmina is built on a foundation of integrity, teamwork, and innovation. We prioritize a collaborative and inclusive workplace where employees are encouraged to grow and contribute to the company's mission. Our dedication to social responsibility and ethical governance reflects our commitment to making a positive impact on the communities we serve.

## Careers at Sanmina
Join our dynamic team! Sanmina offers a wide range of career opportunities across various domains, from engineering and project management to operational roles. We believe in fostering talent and providing employees with ongoing training and development, making it a rewarding workplace for passionate individuals seeking to make their mark in advanced manufacturing.

## Customer Commitment
Our customers range from leading companies in the medical, automotive, and communications sectors to cutting-edge aerospace contractors. By leveraging our global footprint and advanced technology, we deliver innovative solutions that enhance our clients' competitiveness in their respective markets.

## Environmental Commitment
Sanmina is committed to sustainability and complies with all environmental regulations, aiming to minimize our footprint while fostering a culture of responsibility. We continuously seek to improve our practices to protect the planet.

## Connect with Us
Explore opportunities to partner, invest, or start your career with Sanmina. For more information about our solutions, culture, or job openings, visit our website or contact us directly.

## Conclusion
Sanmina Corporation is not just a manufacturer; we are your strategic partner in success. With our end-to-end solutions, dedicated workforce, and commitment to quality, we aim to exceed your expectations and help you achieve your goals.

---

For further inquiries, please reach out to us through our website or at our global locations. We look forward to collaborating with you!

---

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")